# A certified lower bound for the growth constant

This notebook accompanies *Asymptotic Growth of the Number of Rubik's Snake Shapes*, Section IV-A. It is independent of the unpublished research directory and of every other notebook. Run from this directory, `rubiks-snake/`, or the repository root. Dependencies: the standard library, `../rubiks_snake.py`, NumPy, Numba, and SciPy, plus a Jupyter runtime.

A slab block has incoming and outgoing $+x$ edges. Its internal path has $\ell$ edges and stays in $0\leq x\leq d-1$. The entry and exit wedges are included in the collision check. `raw[d][ell]` counts these blocks; their **total block length is $j=\ell+1$**, including the outgoing edge.

Raw blocks of different widths cannot simply be added: a path may have several parsings. Unique cutting at all separating $+x$ edges gives $B(X,Z)=1/(1-I(X,Z))$. We extract $I_{d,j}$ by exact integer convolution, then sum the available irreducible counts. The cutoffs must be nonincreasing in $d$, so every coefficient needed in that convolution is known. Only afterwards are unknown nonnegative irreducibles omitted.

If `counts[ell]` is the resulting count, $\mu\geq\alpha$, where $\sum_\ell \text{counts}[\ell]\alpha^{-(\ell+1)}=1$. For $L=\text{len(counts)}$, the sign of $p^L-\sum_\ell\text{counts}[\ell]p^{L-\ell-1}D^{\ell+1}$ proves $p/D<\mu$. The bound function uses integer bisection, not a floating-point root estimate.

**Reproduction policy.** The saved computation outputs below were reproduced from these cell sources with the repository interpreter on 2026-09-19 (Python 3.14.6, NumPy 2.5.3, Numba 0.67.0, SciPy 1.18.1). Execution counts remain null because validation used a script, not an editor kernel. Rerunning replaces the outputs and regenerates all counts. Timings include Numba compilation on the first call. Enumeration is exponential; improve bounds by increasing the cutoffs gradually. Integer counters raise on overflow rather than silently wrapping.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
started = perf_counter()
cwd = Path.cwd()
candidates = (cwd, cwd / 'rubiks-snake', cwd.parent, cwd.parent / 'rubiks-snake')
snake_dir = next((p for p in candidates if (p / 'rubiks_snake.py').is_file()), None)
if snake_dir is None:
    raise FileNotFoundError('Run from asymptotic-analysis, rubiks-snake, or the repository root')
sys.path.insert(0, str(snake_dir.resolve()))
import numpy as np
import numba
import scipy
from rubiks_snake import slab_counts, irreducible_slab_counts, renewal_lower_bound, renewal_polynomial_value
print(f'Python {sys.version.split()[0]}; NumPy {np.__version__}; Numba {numba.__version__}; SciPy {scipy.__version__}')
print(f'Setup: {perf_counter() - started:.3f} s')

In [ ]:
def get_bound(cutoffs, denominator=10**9):
    """Enumerate, extract irreducibles, and return a certified rational lower bound."""
    if not cutoffs or sorted(cutoffs) != list(range(1, len(cutoffs) + 1)):
        raise ValueError('Use consecutive progress values d=1,...,D')
    limits = [cutoffs[d] for d in sorted(cutoffs)]
    if any(not isinstance(k, int) or k < 1 for k in limits) or limits != sorted(limits, reverse=True):
        raise ValueError('Internal-length cutoffs must be positive and nonincreasing')
    started = perf_counter()
    raw = {}
    for d in sorted(cutoffs):
        row_started = perf_counter()
        raw[d] = slab_counts(d - 1, cutoffs[d])
        print(f'd={d}, K={cutoffs[d]}: {perf_counter() - row_started:.3f} s')
    counts = irreducible_slab_counts(raw)
    q = renewal_lower_bound(counts, denominator)
    sign = renewal_polynomial_value(counts, q.numerator, q.denominator)
    assert sign < 0
    elapsed = perf_counter() - started
    print(f'mu > {q} = {float(q):.9f}; exact polynomial sign < 0; total {elapsed:.3f} s')
    return dict(bound=q, raw=raw, counts=counts, polynomial_value=sign, seconds=elapsed)

## Quick examples, then richer block families

Progress one means width zero: alternating-axis transverse runs in a plane. The later examples include progressively wider slabs. Increasing a cutoff adds genuine irreducible blocks, never duplicate parsings.

In [ ]:
SMALL_CASES = [{1: 4}, {1: 8}, {1: 12}, {1: 16, 2: 12, 3: 10}]
small_results = [get_bound(cutoffs) for cutoffs in SMALL_CASES]

d=1, K=4: 0.237 s
mu > 1518630369/500000000 = 3.037260738; exact polynomial sign < 0; total 0.237 s
d=1, K=8: 0.000 s
mu > 783771809/250000000 = 3.135087236; exact polynomial sign < 0; total 0.000 s
d=1, K=12: 0.000 s
mu > 3143484601/1000000000 = 3.143484601; exact polynomial sign < 0; total 0.000 s
d=1, K=16: 0.001 s
d=2, K=12: 0.013 s
d=3, K=10: 0.004 s
mu > 1654667871/500000000 = 3.309335742; exact polynomial sign < 0; total 0.018 s


## Published bound

Change only `PUBLISHED_CUTOFFS` to extend the calculation, keeping the cutoffs nonincreasing. Adding a new key `4` includes width-three blocks, provided its cutoff does not exceed that of key `3`. All rows below are generated, not loaded from archived tables. The following reproduced output gives the reference coefficients; it is not an input to the code.

In [ ]:
PUBLISHED_CUTOFFS = {1: 28, 2: 20, 3: 18}
published = get_bound(PUBLISHED_CUTOFFS)
for d, row in published['raw'].items():
    print(f'd={d}, K={len(row)-1}: {row}')
print('Irreducible counts by internal length:', published['counts'])
plane_bound = renewal_lower_bound(published['raw'][1])
print(f'Plane-only mu > {plane_bound} = {float(plane_bound):.9f}')
print('Exact scaled polynomial value:', published['polynomial_value'])

d=1, K=28: 0.889 s
d=2, K=20: 47.089 s
d=3, K=18: 36.425 s
mu > 3400034903/1000000000 = 3.400034903; exact polynomial sign < 0; total 84.404 s
d=1, K=28: [0, 4, 8, 16, 24, 40, 72, 136, 224, 392, 712, 1272, 2168, 3840, 6832, 12112, 20904, 36856, 65192, 115096, 199368, 350696, 618032, 1087696, 1887888, 3314376, 5825784, 10230736, 17775440]
d=2, K=20: [0, 0, 0, 16, 64, 192, 448, 1096, 2960, 8688, 25264, 71768, 199984, 553568, 1536880, 4276240, 11894352, 33015408, 91581712, 253615768, 702030784]
d=3, K=18: [0, 0, 0, 0, 0, 64, 384, 1536, 4736, 13760, 41088, 129536, 412160, 1293608, 4005936, 12395208, 38595792, 120780664, 378267416]
Irreducible counts by internal length: [0, 4, 8, 16, 24, 40, 72, 272, 1200, 4984, 17784, 56912, 170728, 506568, 1556752, 5049704, 17048904, 58399712, 199483280, 250948016, 697051336, 350696, 618032, 1087696, 1887888, 3314376, 5825784, 10230736, 17775440]
Plane-only mu > 314438419/100000000 = 3.144384190
Exact scaled polynomial value: -1962266339072738800450361734